# YOLO — Modèle TIREUR (Titulaire du compte)
**Classe unique :** `tireur` (class 0)

**Zone cible :** partie centrale-basse du chèque, sous le bénéficiaire, ligne « Titulaire du compte »

> Annotez le nom manuscrit du tireur uniquement (pas l'étiquette imprimée).

In [ ]:
!pip install ultralytics --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

ZIP_PATH    = '/content/drive/MyDrive/tunicheque/dataset_tireur.zip'
DATASET_DIR = '/content/dataset_tireur'
SAVE_DIR    = '/content/drive/MyDrive/tunicheque/models/yolo_tireur'

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,    exist_ok=True)
print('OK — chemins configurés')

In [ ]:
import zipfile, pathlib

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DATASET_DIR)

for p in sorted(pathlib.Path(DATASET_DIR).rglob('*'))[:30]:
    print(p)

In [ ]:
import pathlib, shutil, random

src = pathlib.Path(DATASET_DIR)
images = sorted(src.rglob('*.jpg')) + sorted(src.rglob('*.png'))
print(f'{len(images)} images trouvées')

random.seed(42)
random.shuffle(images)
split = int(len(images) * 0.8)
train_imgs, val_imgs = images[:split], images[split:]

for split_name, split_list in [('train', train_imgs), ('val', val_imgs)]:
    img_dir = src / split_name / 'images'
    lbl_dir = src / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img in split_list:
        shutil.copy(img, img_dir / img.name)
        lbl = img.with_suffix('.txt')
        if lbl.exists():
            shutil.copy(lbl, lbl_dir / lbl.name)

print(f'Train : {len(train_imgs)} | Val : {len(val_imgs)}')

In [ ]:
yaml_content = f"""path: {DATASET_DIR}
train: train/images
val:   val/images

nc: 1
names:
  0: tireur
"""

yaml_path = '/content/tireur.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(yaml_content)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data    = yaml_path,
    epochs  = 120,          # un peu plus long : zone petite et variable
    imgsz   = 640,
    batch   = 8,
    patience= 25,
    name    = 'yolo_tireur',
    project = '/content/runs',
    exist_ok= True,
    device  = 0,
    augment = True,
    degrees = 5,
    flipud  = 0.0,
    fliplr  = 0.0,
    scale   = 0.4,
    shear   = 2,
    perspective = 0.0003,   # légère distorsion pour chèques tenus à la main
)

print('Entraînement terminé !')

In [ ]:
import shutil, pathlib

best = pathlib.Path('/content/runs/yolo_tireur/weights/best.pt')
dest = pathlib.Path(SAVE_DIR) / 'best.pt'
shutil.copy(best, dest)
print(f'best.pt sauvegardé → {dest}')

In [ ]:
from ultralytics import YOLO
import pathlib

model = YOLO(str(pathlib.Path(SAVE_DIR) / 'best.pt'))

TEST_IMAGE = '/content/drive/MyDrive/tunicheque/test_cheque.jpg'

results = model(TEST_IMAGE, conf=0.25, verbose=True)
for r in results:
    print(f'{len(r.boxes)} détection(s)')
    for b in r.boxes:
        x1,y1,x2,y2 = [int(v) for v in b.xyxy[0].tolist()]
        print(f'  conf={float(b.conf[0]):.3f}  bbox=({x1},{y1})→({x2},{y2})  taille={x2-x1}x{y2-y1}')